# Quadruped Locomotion (Go2): Training (Fine-tuning)

이 노트북은 `go2_locomotion_basic.ipynb` 와 동일한 Colab 환경에서,
**env3 보상으로 학습해 둔 모델(`models/pretrained_env3`)을 불러와 1,600 step 만 추가 학습**한 뒤,
학습한 모델로 추론하여 보행 영상을 생성·재생합니다.

Colab 에서 전체 학습은 시간이 오래 걸리므로, 이어학습(fine-tuning)
방식으로 **학습 → 추론 → 영상** 파이프라인 전체를 빠르게 체험하는 것이 목적입니다.

학습/추론에 사용하는 보상 설정은 `src/envs3.yaml` 입니다.

> 런타임 → 런타임 유형 변경 → **T4 GPU** 로 설정 후 실행하세요.


---

## 0. 환경 설정

GitHub 레포지토리를 clone 하고, MuJoCo / Stable-Baselines3 등 의존성을 설치합니다.
Colab 이면 `/content` 를, 아니면 현재 작업 디렉터리를 기준 경로로 사용합니다.

> numpy 2.x 호환 스택을 사용하므로 **런타임 재시작이 필요 없습니다.** [런타임 > 모두 실행]으로 한 번에 진행됩니다.


In [ ]:
# 1) Clone repository
import os, sys

# Detect Colab by availability of /content or google.colab.
try:
    import google.colab  # noqa: F401
    in_colab = True
except Exception:
    in_colab = os.path.isdir("/content")

try:
    base_dir
except NameError:
    base_dir = "/content" if in_colab else os.getcwd()
os.chdir(base_dir)

repo_dir = os.path.join(base_dir, "RL_tutorial")

print(f"Base directory: {base_dir}")
print(f"Repo directory: {repo_dir}")

if not os.path.isdir(repo_dir):
  !git clone https://github.com/agbread/RL_tutorial.git
else:
  print("Repo exists - 최신 코드로 갱신합니다 (git fetch + reset)")
  !cd "{repo_dir}" && git fetch -q origin && git reset -q --hard origin/main

os.chdir(repo_dir)
print("Current Directory: ", os.getcwd())

In [ ]:
# 2) Install dependencies
# numpy2 호환 스택 → Colab 기본 numpy(2.x)를 그대로 사용하므로 런타임 재시작이 필요 없습니다.
!apt-get -qq install -y libosmesa6 libgl1-mesa-glx  # osmesa(CPU) 렌더링용
!pip install -q "stable-baselines3==2.8.0" "gymnasium==1.2.3" "mujoco==3.8.0" "imageio[ffmpeg]" tensorboard pygments
!pip uninstall -y -q gym shimmy 2>/dev/null  # 구 gym 제거 (numpy2 충돌·렌더 크래시 유발 가능, 우리는 gymnasium만 사용)

In [ ]:
# 3) 의존성 설치 후 환경 설정
import os, sys
import yaml
import torch  # ⚠️ mujoco 보다 먼저 import! (torch↔mujoco 네이티브 라이브러리 로드 순서가 바뀌면 import에서 멈춤/크래시)

sys.path.insert(0, os.path.join(repo_dir, "src"))
os.environ["MUJOCO_GL"] = "egl"  # Colab GPU 렌더링 (참조 노트북과 동일). torch를 먼저 import하면 안정적

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import HtmlFormatter
import inspect

def _render_code(code, title="code", max_height=400, bg="transparent", indent=16):
    style_name = "native" if in_colab else "friendly"
    formatter = HtmlFormatter(style=style_name, noclasses=True, linenos="inline")
    html = highlight(code, PythonLexer(), formatter)
    css = """
    <style>
    .highlight pre { margin: 0; text-align: left; }
    </style>
    """
    return HTML(f"""
    {css}
    <details>
      <summary>{title}</summary>
      <div style="margin-top:8px; margin-left:{indent}px; max-height:{max_height}px; overflow:auto; border:1px solid #ddd; padding:10px; background:{bg};">
        {html}
      </div>
    </details>
    """)


def show_code(path, max_height=400, bg="transparent"):
    code = Path(path).read_text()
    return _render_code(code, title=str(path), max_height=max_height, bg=bg)

def show_func(obj, max_height=400, bg="transparent"):
    code = inspect.getsource(obj)
    return _render_code(code, max_height=max_height, bg=bg)

---

## 1. 설정 파일 살펴보기

학습에 사용하는 주요 설정/구현 파일을 노트북에서 바로 열어봅니다.

- **`src/params.yaml`** — PPO 하이퍼파라미터, 병렬 환경 수, 총 timestep, 평가 주기 등 학습 설정
- **`src/envs.yaml`** — 보상/패널티 가중치, 명령 속도 범위, 보행(gait) 패턴, 종료 조건
- **`src/mdp/reward.py`** — 위 가중치가 실제로 계산되는 보상 함수 구현


In [ ]:
# 설정/구현 파일 펼쳐 보기 (각 제목을 클릭하면 코드가 펼쳐집니다)
display(show_code(f"{repo_dir}/src/params.yaml"))
display(show_code(f"{repo_dir}/src/envs3.yaml"))
display(show_code(f"{repo_dir}/src/mdp/reward.py", max_height=600))

---

## 2. Go2 MuJoCo 환경 생성

`Go2MujocoEnv`(position 제어 전용, `unitree_go2/scene_position.xml`)를 생성하고
관측(observation)·행동(action) 공간을 확인합니다. `params.yaml` 도 함께 로드합니다.


In [ ]:
import numpy as np
import src.go2_mujoco_env as go2_env

# 학습 설정 로드
policy_cfg_path = f"{repo_dir}/src/params.yaml"
with open(policy_cfg_path, "r", encoding="utf-8") as f:
    policy_cfg = yaml.safe_load(f)

# 환경 생성 및 공간 확인
env = go2_env.Go2MujocoEnv(prj_path=repo_dir, render_mode=None)
obs, info = env.reset()
print("n_envs       :", policy_cfg["n_envs"])
print("batch_size   :", policy_cfg["policy"]["batch_size"])
print("Observation shape:", np.array(obs).shape)
print("Action space     :", env.action_space)
print("Observation space:", env.observation_space)
env.close()

---

## 3. Pretrained 모델 불러와 1,600 step 추가 학습

전체 학습은 Colab 에서 비용이 크므로, **env3 모델을 불러와
`reset_num_timesteps=False` 로 1,600 step 만 이어서 학습**합니다.

- `PRETRAINED_MODEL_PATH` : 불러올 모델 경로. 기본값은
  `models/pretrained_env3/best_model.zip` 입니다. 다른 모델로 바꾸려면
  이 한 줄만 수정하면 됩니다.
- `ENV_CFG_PATH` : 학습에 사용할 보상 설정. 기본값 `src/envs3.yaml`.
- `ADDITIONAL_TIMESTEPS` : 추가 학습 step (기본 1,600)
- 학습 결과는 `models/<날짜시각>-finetune10k/` 에 저장됩니다
  (`best_model.zip`, 체크포인트, `final_model.zip`).
- numpy 버전 불일치로 인한 로드 실패를 막기 위해 `PPO.load(custom_objects=...)` 를 사용합니다.


In [ ]:
import time, gc
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import (
    EvalCallback, CheckpointCallback, CallbackList,
)
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv

import src.go2_mujoco_env as go2_env
from src.utils.reward_logging_callback import RewardLoggingCallback

# ===== 설정 (필요시 수정) =====
# env3 보상으로 학습한 모델을 출발점으로 사용.
PRETRAINED_MODEL_PATH = f"{repo_dir}/models/pretrained_env3/best_model.zip"
ADDITIONAL_TIMESTEPS = 1600  # n_steps(400)×N_ENVS(4)=1600 → PPO rollout 1회라 가장 빠름
N_ENVS = 4  # Colab 메모리 안전값 (12는 OOM)
SEED = policy_cfg["seed"]

assert os.path.exists(PRETRAINED_MODEL_PATH), (
    f"pretrained 모델을 찾을 수 없습니다: {PRETRAINED_MODEL_PATH}\n"
    f"PRETRAINED_MODEL_PATH 를 존재하는 .zip 경로로 수정하세요."
)

model_dir = f"{repo_dir}/models"
log_dir = f"{repo_dir}/logs"
os.makedirs(model_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

ENV_CFG_PATH = f"{repo_dir}/src/envs3.yaml"   # 이 보상 설정으로 학습
train_env_kwargs = {"prj_path": repo_dir, "cfg_path": ENV_CFG_PATH}
vec_env = make_vec_env(
    go2_env.Go2MujocoEnv, env_kwargs=train_env_kwargs,
    n_envs=N_ENVS, seed=SEED, vec_env_cls=SubprocVecEnv,
)
eval_env = make_vec_env(
    go2_env.Go2MujocoEnv, env_kwargs=train_env_kwargs,
    n_envs=1, seed=SEED + 10_000, vec_env_cls=DummyVecEnv,
)

run_name = time.strftime("%Y-%m-%d_%H-%M-%S") + "-finetune10k"
model_path = f"{model_dir}/{run_name}"
os.makedirs(model_path, exist_ok=True)
print("저장 위치:", model_path)

# numpy 버전 불일치로 space 역직렬화가 실패할 수 있어 현재 env 의 space 로 대체
_dummy = go2_env.Go2MujocoEnv(prj_path=repo_dir, cfg_path=ENV_CFG_PATH, render_mode=None)
custom_objects = {
    "observation_space": _dummy.observation_space,
    "action_space": _dummy.action_space,
    "lr_schedule": lambda _: policy_cfg["policy"]["learning_rate"],
    "clip_range": lambda _: policy_cfg["policy"]["clip_range"],
}
_dummy.close()

checkpoint_callback = CheckpointCallback(
    save_freq=max(policy_cfg["policy"]["n_steps"] * policy_cfg["log"]["interval"] // N_ENVS, 1),
    save_path=model_path, name_prefix="model",
    save_replay_buffer=False, save_vecnormalize=False,
)
eval_callback = EvalCallback(
    eval_env, best_model_save_path=model_path, log_path=log_dir,
    eval_freq=max(policy_cfg["eval_freq"] // N_ENVS, 1),
    n_eval_episodes=5, deterministic=True, render=False,
)
callbacks = CallbackList([eval_callback, checkpoint_callback, RewardLoggingCallback()])

print(f"Loading pretrained model from {PRETRAINED_MODEL_PATH}")
model = PPO.load(
    PRETRAINED_MODEL_PATH, env=vec_env, custom_objects=custom_objects,
    verbose=1, tensorboard_log=log_dir,
)
# 학습률을 새로 적용 (스케줄 재생성)
model.learning_rate = policy_cfg["policy"]["learning_rate"]
model._setup_lr_schedule()

model.learn(
    total_timesteps=ADDITIONAL_TIMESTEPS,
    reset_num_timesteps=False,   # pretrained 의 step 카운트를 이어감
    progress_bar=True,
    tb_log_name=run_name,
    callback=callbacks,
)
model.save(f"{model_path}/final_model")
print("최종 모델 저장:", f"{model_path}/final_model.zip")

vec_env.close()
eval_env.close()
del model
gc.collect()

---

## 4. 학습한 모델로 추론 → 보행 영상 생성

위에서 추가 학습해 저장한 모델을 불러와, 직진 command `[vx, vy, wz] = [0.9, 0, 0]`
으로 롤아웃하면서 프레임을 모아 mp4 로 저장합니다. (제어 50Hz, 영상 10FPS)


In [ ]:
# 렌더링은 별도 프로세스에서 수행하고, 불안정한 환경(예: Colab)이면 사전 렌더링 영상으로 대체합니다.
import subprocess, sys

eval_model_path = f"{model_path}/best_model.zip"
if not os.path.exists(eval_model_path):
    eval_model_path = f"{model_path}/final_model.zip"
print("추론 모델:", eval_model_path)

video_path = f"{model_path}/rollout_{run_name}.mp4"
base = [
    sys.executable, "-u", f"{repo_dir}/src/render_rollout.py",
    "--prj", repo_dir, "--model", eval_model_path, "--cfg", ENV_CFG_PATH,
    "--out", video_path, "--command", "0.9", "0.0", "0.0",
    "--max_time_s", str(policy_cfg["test"]["max_time_step_s"]),
    "--width", "320", "--height", "240", "--camera", "tracking",
]
ok = False
for gl in ["egl", "osmesa"]:
    res = subprocess.run(base + ["--gl", gl], capture_output=True, text=True)
    if res.returncode == 0 and os.path.exists(video_path):
        print(f"라이브 렌더 성공 ({gl}):", video_path); ok = True; break

if not ok:
    fb = f"{repo_dir}/models/_pretrained_rollouts/env3.mp4"
    if os.path.exists(fb):
        print("ℹ️ 라이브 렌더 미지원 환경 → 사전 렌더링 영상 표시:", fb)
        video_path = fb
    else:
        print("⚠️ 렌더 실패 + 대체 영상 없음")

---

## 5. 영상 재생

생성된 mp4 를 노트북에서 바로 재생합니다.


In [ ]:
from IPython.display import Video, display

display(Video(video_path, embed=True, html_attributes="controls autoplay loop"))

---

## 6. TensorBoard 로 학습 로그 보기

PPO 학습 중 기록된 보상/손실 곡선을 TensorBoard 로 확인합니다.
런(run)별로 `logs/` 아래에 저장되며, 셀을 실행하면 노트북 안에 대시보드가 뜹니다.

> `logs/pretrained_env*` 에 사전학습(0→~2M step) 로그가 포함돼 있어, **사전학습 전체 곡선 + 이어학습 구간**이 같은 step 축 위에 함께 표시됩니다.


In [ ]:
# 학습 로그 시각화 (Colab 인라인 TensorBoard)
import os
os.chdir(repo_dir)          # logs 상대경로 기준 보장
%load_ext tensorboard
%tensorboard --logdir logs